# Gold — dim_comment_scd1 (SCD 1, poc method)

One row per YouTube comment; `like_count` / `reply_count` = latest read (type 1).
Method (poc `scd_type_1`, helper `_scd`): **changed keys** in the window → re-read their full current rows → metadata `etl_date` / `etl_by` → `add_checksums` → **full**: overwrite / **incremental**: MERGE on `primary_checksum`, update only when `record_checksum` differs → Unknown row → grain check. Widgets: `load_type`, `from_date`, `to_date` (Vietnam days, empty = yesterday .. tomorrow). Runs in task `gold_dims_scd1` (nothing depends on it); the live dim is unchanged.

**Changed keys** = comments read in the window (`stg_comments.snapshot_date`) **∪** comments already in the table whose `video_key` or `commenter_key` is still −1 (late-arriving dimension: the video / author reached its dim after the comment). Values from `stg_comments_latest`; `video_key` / `commenter_key` looked up in `dim_video_scd1` / `dim_commenter_scd1` (−1 when missing), so this notebook runs after those two.

In [ ]:
%run ./_scd

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow (today's reads included)
SILVER, GOLD = dbutils.widgets.get("silver"), dbutils.widgets.get("gold")
LOAD_TYPE = dbutils.widgets.get("load_type")
FROM_DATE, TO_DATE = default_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))
FULL = LOAD_TYPE == "full"
print("full: every key" if FULL else f"incremental: keys changed in [{FROM_DATE}, {TO_DATE}) (Vietnam days)")

In [ ]:
TABLE = f"{GOLD}.dim_comment_scd1"
latest = spark.table(f"{SILVER}.stg_comments_latest").where("comment_id IS NOT NULL")
if not FULL:
    unresolved = (spark.table(TABLE).where("comment_key <> -1 AND (video_key = -1 OR commenter_key = -1)").select("comment_id")
                  if spark.catalog.tableExists(TABLE) else None)
    frames = [spark.table(f"{SILVER}.stg_comments").where(in_window(F.col("snapshot_date"), FROM_DATE, TO_DATE)).select("comment_id")]
    keys = changed_keys(*(frames + ([unresolved] if unresolved is not None else [])))
    latest = latest.join(keys, "comment_id", "left_semi")

videos = spark.table(f"{GOLD}.dim_video_scd1").where("video_key <> -1").select("video_id", "video_key")
authors = spark.table(f"{GOLD}.dim_commenter_scd1").where("commenter_key <> -1").select("commenter_nk", "commenter_key")
src = (latest
       .withColumn("commenter_nk", F.coalesce(F.col("author_channel_id"), F.concat(F.lit("name:"), F.col("author_display_name"))))
       .join(videos, "video_id", "left").join(authors, "commenter_nk", "left")
       .select(F.xxhash64("comment_id").alias("comment_key"), "comment_id",
               F.coalesce("video_key", F.lit(-1)).cast("bigint").alias("video_key"), "video_id",
               F.coalesce("commenter_key", F.lit(-1)).cast("bigint").alias("commenter_key"), "author_display_name",
               F.col("published_at").alias("posted_at"),
               F.date_format(F.from_utc_timestamp("published_at", TZ), "yyyyMMdd").cast("int").alias("posted_date_key"),
               F.col("text").alias("comment_text"), "like_count", F.col("total_reply_count").cast("int").alias("reply_count"),
               "created_at", F.col("snapshot_ts").alias("modified_at"))
       .withColumn("etl_date", F.current_timestamp())
       .withColumn("etl_by", F.lit("Databricks")))
src = add_checksums(src, ["comment_id"], exclude=["comment_key", "created_at", "modified_at"])

In [ ]:
load_scd1(src, TABLE, """
        comment_key BIGINT, comment_id STRING, video_key BIGINT, video_id STRING, commenter_key BIGINT,
        author_display_name STRING, posted_at TIMESTAMP, posted_date_key INT, comment_text STRING, like_count BIGINT,
        reply_count INT, created_at TIMESTAMP, modified_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING""",
    "Comment dimension, SCD 1 (poc checksum method). like_count / reply_count = latest read. comment_key = xxhash64(comment_id), -1 = Unknown.",
    ["comment_id"], LOAD_TYPE, {"comment_key": "-1", "comment_text": "'Unknown'", "video_key": "-1", "commenter_key": "-1",
                                "etl_date": "current_timestamp()", "etl_by": "'Databricks'"})